# NATMO and GSI map evidence; borehole crosswalk

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Collects map evidence on the regional geology: the GSI catalogue, the NATMO district planning map (thematic panels and legends), a crosswalk between NATMO/GSI units and the borehole lithology, the Plate IV legend and CRS search, and the maps in the Rajasthan aquifer mapping report (in progress).

> ℹ️ The last two cells (download and extraction of the Rajasthan aquifer mapping report maps) ended with errors in the source notebook — this part is still in progress.

| | |
|---|---|
| **Reads** | `MODFLOW_Alwar/…` (NATMO and GSI images), `Step7ID_Interval_Conceptual_Interpretation.csv`, the DEM, report PDFs |
| **Writes** | `MODFLOW_Alwar/…` (`NATMO_…`, `G21F7_…` to `G21F13_…` registers) |
| **Cells** | 25 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. GSI catalogue
- 2. NATMO district planning map
- 3. Borehole lithology inventory and crosswalk
- 4. GSI Plate IV legend, CRS and reference layers
- 5. Rajasthan aquifer mapping report maps (in progress)

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["pymupdf", "pillow", "geopandas", "rasterio", "requests", "openpyxl"], modflow=False)
del TOKEN, URL


## 1. GSI catalogue

**[orig #330]** Search the GSI data.gov.in catalogue for Alwar records  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path
import pandas as pd

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

FOLDER = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/GSI_Original_Source"
)

FILE = FOLDER / "600_DRM_data_gov_in_11072022_SUBMITTED.csv"

if not FILE.exists():
    raise FileNotFoundError(f"CSV not found: {FILE}")

# Read catalogue
df = pd.read_csv(FILE, encoding="utf-8", low_memory=False)

print("Catalogue shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

print("\nFirst five records:")
display(df.head())

# Search all text columns for Alwar
text_cols = df.select_dtypes(include=["object", "string"]).columns

mask = pd.Series(False, index=df.index)

for col in text_cols:
    mask |= df[col].astype(str).str.contains(
        "Alwar", case=False, na=False
    )

alwar = df.loc[mask].copy()

print("\nAlwar-related records:", len(alwar))
display(alwar)

# Save extracted records
OUT = FOLDER / "Step7ID4G21C_Alwar_GSI_Catalogue_Records.csv"
alwar.to_csv(OUT, index=False)

print("\nSaved:", OUT)

**[orig #331]** Show the matching catalogue records  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

FOLDER = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/GSI_Original_Source"
)

df = pd.read_csv(
    FOLDER / "600_DRM_data_gov_in_11072022_SUBMITTED.csv",
    low_memory=False
)

alwar = df[
    df["DISTRICT NAME"].astype(str).str.strip().str.lower() == "alwar"
].copy()

for i, row in alwar.iterrows():
    print("\nCATALOGUE ROW:", i)
    for col in [
        "MAP NAME", "YEAR", "SCALE", "RESTRICTED",
        "STATE NAMES", "DISTRICT NAME", "TOPOSHEET NO", "URL"
    ]:
        print(f"{col}: {row[col]}")

    print("\nURL length:", len(str(row["URL"])))

## 2. NATMO district planning map

**[orig #334]** Open the NATMO district planning map  
<sub>✅ ran OK</sub>

In [ ]:
from PIL import Image
from pathlib import Path
from IPython.display import display

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference"
)

TARGET = BASE / "G21F6_Detailed_Planning_Map_Extraction" / \
         "NATMO_Alwar_Detailed_Planning_Map_Refined.png"

# Prefer refined image; otherwise use the original downloaded map.
if TARGET.exists():
    P = TARGET
else:
    P = BASE / "NATMO_Alwar_District_Planning_Map.jpg"

if not P.exists():
    raise FileNotFoundError(
        "Neither refined nor original NATMO image was found. "
        "Run Step 1 and inspect the listed paths."
    )

img = Image.open(P)

print("Selected image:", P)
print("Dimensions:", img.size)
print("Format:", img.format)

display(img)

**[orig #335]** G21F6 — extract the NATMO thematic panels  
<sub>✅ ran OK</sub>

In [ ]:

# ============================================================
# G21F6 – NATMO THEMATIC PANEL EXTRACTION
# Original image: 2048 x 1291 pixels
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

# 1. Input image
base = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference"
)

image_path = base / "NATMO_Alwar_District_Planning_Map.jpg"

if not image_path.exists():
    raise FileNotFoundError(f"Original NATMO image not found: {image_path}")

img = Image.open(image_path).convert("RGB")
W, H = img.size

print("Original image:", img.size)
print("Image format:", Image.open(image_path).format)

# 2. Output directory
out_dir = base / "G21F6_Detailed_Planning_Map_Extraction" / "Thematic_Panels"
out_dir.mkdir(parents=True, exist_ok=True)

# 3. Preliminary crop coordinates (left, upper, right, lower)
# These are inspection crops, NOT georeferencing coordinates.
panels = {
    "Rocks_and_Minerals": (1400, 365, 1750, 730),
    "Relief_and_Slope": (1740, 365, 2048, 730),
    "Irrigation_and_Hydrogeology": (1110, 720, 1750, 1105)
}

# 4. Extract, save and display
records = []

for name, bounds in panels.items():
    x1, y1, x2, y2 = bounds

    # Check crop coordinates
    assert 0 <= x1 < x2 <= W
    assert 0 <= y1 < y2 <= H

    crop = img.crop(bounds)
    output_path = out_dir / f"NATMO_{name}.png"
    crop.save(output_path)

    records.append({
        "panel": name,
        "x1": x1, "y1": y1,
        "x2": x2, "y2": y2,
        "width": crop.width,
        "height": crop.height,
        "output_file": str(output_path)
    })

    print(f"\n{name}: {crop.size}")
    display(crop)

# 5. Save extraction register
register = pd.DataFrame(records)
register_path = out_dir / "NATMO_Panel_Extraction_Register.csv"
register.to_csv(register_path, index=False)

print("\nExtraction completed.")
print("Output directory:", out_dir)
print("Register:", register_path)


**[orig #336]** Corrected panel crop windows  
<sub>✅ ran OK</sub>

In [ ]:

from PIL import Image
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Original NATMO image
base = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference"
)

image_path = base / "NATMO_Alwar_District_Planning_Map.jpg"

img = Image.open(image_path).convert("RGB")
W, H = img.size

print("Original dimensions:", W, "x", H)
assert img.size == (2362, 1490), "Unexpected image dimensions. Check original."

# Output directory
out_dir = (
    base / "G21F6_Detailed_Planning_Map_Extraction"
    / "Thematic_Panels_Corrected"
)
out_dir.mkdir(parents=True, exist_ok=True)

# Corrected preliminary crop windows:
# (left, top, right, bottom)
panels = {
    "Rocks_and_Minerals": (1640, 430, 2015, 845),
    "Relief_and_Slope": (2000, 430, 2360, 845),
    "Irrigation_and_Hydrogeology": (1275, 830, 1970, 1270)
}

records = []

for name, bounds in panels.items():
    crop = img.crop(bounds)

    output_path = out_dir / f"NATMO_{name}_corrected.png"
    crop.save(output_path)

    records.append({
        "panel": name,
        "crop_coordinates": str(bounds),
        "width": crop.width,
        "height": crop.height,
        "file": str(output_path)
    })

    print(f"\n{name}: {crop.size}")
    display(crop)

# Save extraction register
pd.DataFrame(records).to_csv(
    out_dir / "NATMO_Corrected_Panel_Register.csv",
    index=False
)

print("\nCorrected crops saved to:", out_dir)


**[orig #337]** NATMO thematic evidence register  
<sub>✅ ran OK</sub>

In [ ]:

import pandas as pd
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference/"
    "G21F6_Detailed_Planning_Map_Extraction/"
    "Thematic_Panels_Corrected"
)

records = [
    {
        "Panel": "Rocks and Minerals",
        "Scale": "1:1,000,000",
        "Observed_information": "Recent Sand; Limestone; Quartzite/Conglomerate; Rhyolite (provisional transcription); mineral occurrence symbols",
        "Research_relevance": "Regional lithological context",
        "Confidence": "Moderate - legend requires higher-resolution verification",
        "Permitted_use": "Supplementary regional geological evidence",
        "Not_permitted": "Direct assignment of hydraulic properties or subsurface layer geometry"
    },
    {
        "Panel": "Irrigation and Hydrogeology",
        "Scale": "1:1,000,000",
        "Observed_information": "Groundwater potential yield classes; water-table contours; groundwater exploitation influence; irrigation sources",
        "Research_relevance": "Regional groundwater and irrigation context",
        "Confidence": "Moderate - detailed legend values require verification",
        "Permitted_use": "Qualitative regional comparison",
        "Not_permitted": "Treating potential classes as measured aquifer parameters"
    },
    {
        "Panel": "Relief and Slope",
        "Scale": "1:1,000,000",
        "Observed_information": "Elevation classes and slope classes in metres per kilometre",
        "Research_relevance": "Regional terrain and drainage context",
        "Confidence": "Moderate - printed class labels require verification",
        "Permitted_use": "Regional terrain interpretation",
        "Not_permitted": "Using map classes as surveyed elevation data"
    }
]

df = pd.DataFrame(records)

output = base / "NATMO_Thematic_Evidence_Register.csv"
df.to_csv(output, index=False)

print("Evidence register created:")
print(output)
display(df)


**[orig #338]** Enlarge the legend windows  
<sub>✅ ran OK</sub>

In [ ]:

from PIL import Image
from pathlib import Path
from IPython.display import display

base = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference"
)

source = base / "NATMO_Alwar_District_Planning_Map.jpg"

img = Image.open(source).convert("RGB")
print("Original image dimensions:", img.size)

out = (
    base / "G21F6_Detailed_Planning_Map_Extraction"
    / "Legend_Verification"
)
out.mkdir(parents=True, exist_ok=True)

# Preliminary legend windows in ORIGINAL image coordinates.
# These are inspection windows; adjust if any text is clipped.
legends = {
    "Rocks_Minerals_Legend": (1875, 490, 2010, 680),
    "Relief_Slope_Legend": (2225, 490, 2360, 680),
    "Irrigation_Hydrogeology_Legend": (1795, 965, 1970, 1150)
}

for name, bounds in legends.items():
    crop = img.crop(bounds)

    # Enlarge for visual reading; original crop remains unchanged.
    enlarged = crop.resize(
        (crop.width * 4, crop.height * 4),
        Image.Resampling.LANCZOS
    )

    save_path = out / f"{name}_4x.png"
    enlarged.save(save_path)

    print(f"\n{name}")
    print("Original crop:", crop.size)
    print("Saved:", save_path)
    display(enlarged)

print("\nLegend extraction completed:", out)


**[orig #339]** Show the full irrigation/hydrogeology panel  
<sub>✅ ran OK</sub>

In [ ]:

from PIL import Image
from pathlib import Path
from IPython.display import display

base = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference"
)

img = Image.open(
    base / "NATMO_Alwar_District_Planning_Map.jpg"
).convert("RGB")

# Display the complete irrigation/hydrogeology panel
# so we can locate the full groundwater-potential legend.
crop = img.crop((1275, 820, 2020, 1280))

display(crop.resize(
    (1490, 920),
    Image.Resampling.LANCZOS
))

# Save without modifying the original
out = (
    base / "G21F6_Detailed_Planning_Map_Extraction"
    / "Legend_Verification"
)
out.mkdir(parents=True, exist_ok=True)

crop.save(out / "NATMO_Hydrogeology_Full_Legend_Context.png")

print("Saved full hydrogeology legend context.")


**[orig #340]** Add legend-verification fields → register v2  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd

folder = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference/"
    "G21F6_Detailed_Planning_Map_Extraction/"
    "Thematic_Panels_Corrected"
)

source = folder / "NATMO_Thematic_Evidence_Register.csv"

df = pd.read_csv(source)

# Add detailed legend-verification fields
df["Verified_legend_details"] = ""
df["Uncertainty_notes"] = ""
df["Updated_interpretation"] = ""

df.loc[df["Panel"] == "Rocks and Minerals", [
    "Verified_legend_details", "Uncertainty_notes",
    "Updated_interpretation"
]] = [
    "Recent Sand; Limestone; Quartzite-Conglomerate; mineral symbols for Asbestos, Bauxite, Copper, Tin and Steatite",
    "Green unit label appears to read Epidotite; verify against a sharper original",
    "Regional lithological evidence only"
]

df.loc[df["Panel"] == "Relief and Slope", [
    "Verified_legend_details", "Uncertainty_notes",
    "Updated_interpretation"
]] = [
    "Elevation: below 300, 300-450, 450-600, above 600 m. Slope: 150-300, 80-150, 30-80, less than 30 m/km",
    "Printed class labels should be checked against a sharper source before quantitative digitization",
    "Regional terrain and drainage context"
]

df.loc[df["Panel"] == "Irrigation and Hydrogeology", [
    "Verified_legend_details", "Uncertainty_notes",
    "Updated_interpretation"
]] = [
    "Groundwater potential yield in L/s; water-table contours; discontinuous aquifers in porous formations; canals; wells/tubewells; irrigated and unirrigated areas",
    "First yield class appears 4-10 L/s; second yield range unreadable. Water-table contour unit and exact aquifer-description wording need verification",
    "Qualitative regional hydrogeological and groundwater-use context"
]

output = folder / "NATMO_Thematic_Evidence_Register_v2.csv"
df.to_csv(output, index=False)

print("Updated register saved:")
print(output)
display(df[[
    "Panel",
    "Verified_legend_details",
    "Uncertainty_notes",
    "Updated_interpretation"
]])


**[orig #341]** G21F7 — NATMO / GSI / borehole crosswalk  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd

base = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference/"
    "G21F6_Detailed_Planning_Map_Extraction"
)

out_dir = base.parent / "G21F7_NATMO_GSI_Borehole_Comparison"
out_dir.mkdir(parents=True, exist_ok=True)

# Regional NATMO units: preserve the printed map classifications.
units = [
    ("Recent Sand", "Beige", "Regional unconsolidated material"),
    ("Limestone", "Yellow", "Regional sedimentary rock"),
    ("Quartzite-Conglomerate", "Pink", "Combined regional map category"),
    ("Epidotite (provisional transcription)", "Green",
     "Legend spelling requires verification")
]

rows = []

for unit, colour, description in units:
    rows.append({
        "NATMO_unit": unit,
        "NATMO_map_colour": colour,
        "NATMO_description": description,
        "NATMO_source": "NATMO Alwar District Planning Map",
        "NATMO_scale": "1:1,000,000",
        "GSI_corresponding_unit": "Not yet verified",
        "GSI_source_reference": "Pending comparison",
        "Borehole_support": "Not yet evaluated",
        "Borehole_ids": "Pending",
        "Spatial_correlation": "Not evaluated",
        "Confidence": "Unassessed",
        "Conceptual_model_decision": "No approval",
        "Comments": "Regional evidence only; no hydraulic properties inferred"
    })

crosswalk = pd.DataFrame(rows)

csv_path = out_dir / "G21F7_NATMO_GSI_Borehole_Crosswalk.csv"
crosswalk.to_csv(csv_path, index=False)

print("G21F7 crosswalk created:")
print(csv_path)
display(crosswalk)


## 3. Borehole lithology inventory and crosswalk

**[orig #342]** Verify the integrated interval table (`Step7ID_Interval_Conceptual_Interpretation.csv`)  
<sub>✅ ran OK</sub>

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

folder = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input/"
    "Step7I_Conceptual_Model/"
    "Step7ID_Preliminary_Conceptual_Aquifer_System"
)

files = {
    "intervals": "Step7ID_Interval_Conceptual_Interpretation.csv",
    "boreholes": "Step7ID_Borehole_Conceptual_Summary.csv",
    "review": "Step7ID_Conceptual_Review_Register.csv",
    "regional": "Step7ID_Regional_CGWB_Framework.csv",
    "systems": "Step7ID_Conceptual_System_Summary.csv"
}

data = {}

for key, filename in files.items():
    path = folder / filename

    if path.exists():
        df = pd.read_csv(path)
        data[key] = df

        print(f"\n{'='*65}")
        print(key.upper(), ":", filename)
        print("Rows:", len(df))
        print("Columns:", list(df.columns))
        display(df.head(3))
    else:
        print("\nMISSING:", path)

# Verify the main integrated interval table
if "intervals" in data:
    df = data["intervals"]

    print("\n========== DATASET QC ==========")
    print("Total intervals:", len(df))

    # Identify likely borehole ID column
    candidates = [
        c for c in df.columns
        if c.lower() in ["borehole_id", "well_id", "bh_id"]
    ]

    if candidates:
        id_col = candidates[0]
        print("Borehole ID column:", id_col)
        print("Unique boreholes:", df[id_col].nunique())
        print("NHP-133 records:",
              df[id_col].astype(str).str.contains(
                  "NHP-133", case=False, na=False
              ).sum())

    print("\nAvailable lithology/HSG fields:")
    print([
        c for c in df.columns
        if any(k in c.lower()
               for k in ["lith", "hsg", "depth", "elevation",
                         "conceptual", "rock", "material"])
    ])


**[orig #343]** G21F7 — borehole lithology inventory and class summaries  
<sub>✅ ran OK</sub>

In [ ]:

import pandas as pd
from pathlib import Path

# ------------------------------------------------
# INPUT
# ------------------------------------------------
folder = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input/"
    "Step7I_Conceptual_Model/"
    "Step7ID_Preliminary_Conceptual_Aquifer_System"
)

df = pd.read_csv(
    folder / "Step7ID_Interval_Conceptual_Interpretation.csv"
)

out = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference/"
    "G21F7_NATMO_GSI_Borehole_Comparison"
)
out.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# 1. UNIQUE SOURCE LITHOLOGY INVENTORY
# ------------------------------------------------
lithology = (
    df.groupby(
        ["Source_Lithology", "Lithology_Class"],
        dropna=False
    )
    .agg(
        Interval_Count=("Borehole_ID", "size"),
        Borehole_Count=("Borehole_ID", "nunique")
    )
    .reset_index()
    .sort_values("Interval_Count", ascending=False)
)

lithology.to_csv(
    out / "G21F7_Borehole_Lithology_Inventory.csv",
    index=False
)

print("\n1. SOURCE LITHOLOGY INVENTORY")
display(lithology)

# ------------------------------------------------
# 2. LITHOLOGY CLASS DISTRIBUTION
# ------------------------------------------------
class_summary = (
    df.groupby("Lithology_Class", dropna=False)
    .agg(
        Intervals=("Borehole_ID", "size"),
        Boreholes=("Borehole_ID", "nunique")
    )
    .reset_index()
    .sort_values("Intervals", ascending=False)
)

print("\n2. LITHOLOGY CLASS DISTRIBUTION")
display(class_summary)

class_summary.to_csv(
    out / "G21F7_Lithology_Class_Summary.csv",
    index=False
)

# ------------------------------------------------
# 3. HYDROSTRATIGRAPHIC GROUP DISTRIBUTION
# ------------------------------------------------
hsg_summary = (
    df.groupby(
        ["Hydrostratigraphic_Group_ID",
         "Hydrostratigraphic_Group"],
        dropna=False
    )
    .agg(
        Intervals=("Borehole_ID", "size"),
        Boreholes=("Borehole_ID", "nunique")
    )
    .reset_index()
)

print("\n3. HSG DISTRIBUTION")
display(hsg_summary)

# ------------------------------------------------
# 4. CONCEPTUAL SYSTEM DISTRIBUTION
# ------------------------------------------------
system_summary = (
    df.groupby(
        ["Preliminary_Conceptual_System",
         "Local_Evidence_Interpretation"],
        dropna=False
    )
    .agg(
        Intervals=("Borehole_ID", "size"),
        Boreholes=("Borehole_ID", "nunique")
    )
    .reset_index()
)

print("\n4. PRELIMINARY CONCEPTUAL SYSTEMS")
display(system_summary)

system_summary.to_csv(
    out / "G21F7_Conceptual_System_Summary.csv",
    index=False
)

# ------------------------------------------------
# 5. BOREHOLE COORDINATE AND QUALITY SUMMARY
# ------------------------------------------------
coords = (
    df.groupby("Borehole_ID")
    .agg(
        Dataset=("Dataset", "first"),
        Latitude=("Latitude", "first"),
        Longitude=("Longitude", "first"),
        Coordinate_Source=("Coordinate_Source", "first"),
        Coordinate_Status=("Coordinate_Status", "first"),
        Interval_Count=("Source_Row", "size")
    )
    .reset_index()
)

coords.to_csv(
    out / "G21F7_Borehole_Coordinate_Inventory.csv",
    index=False
)

print("\n5. BOREHOLE COORDINATE INVENTORY")
display(coords)

print("\nCoordinate availability:")
print("Total boreholes:", len(coords))
print("Valid coordinate pairs:",
      coords[["Latitude", "Longitude"]].dropna().shape[0])

print("\nAll output files saved to:")
print(out)


**[orig #344]** Literal lithology match audit against the NATMO terms  
<sub>✅ ran OK</sub>

In [ ]:

import pandas as pd
from pathlib import Path

# Load original accepted interval descriptions
folder = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input/"
    "Step7I_Conceptual_Model/"
    "Step7ID_Preliminary_Conceptual_Aquifer_System"
)

df = pd.read_csv(
    folder / "Step7ID_Interval_Conceptual_Interpretation.csv"
)

out = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference/"
    "G21F7_NATMO_GSI_Borehole_Comparison"
)
out.mkdir(parents=True, exist_ok=True)

# Literal search terms: do not silently treat related rocks as equivalent.
search_terms = {
    "Recent Sand": [r"\bsand\b"],
    "Limestone": [r"\blimestone\b"],
    "Quartzite-Conglomerate": [
        r"\bquartzite\b",
        r"\bconglomerate\b"
    ],
    "Epidotite": [r"\bepidotite\b"]
}

records = []

for natmo_unit, patterns in search_terms.items():
    for pattern in patterns:
        mask = df["Source_Lithology"].fillna("").str.contains(
            pattern, case=False, regex=True
        )
        matched = df.loc[mask].copy()

        records.append({
            "NATMO_Unit": natmo_unit,
            "Search_Term": pattern,
            "Matching_Intervals": len(matched),
            "Unique_Boreholes": matched["Borehole_ID"].nunique(),
            "Borehole_IDs": "; ".join(
                sorted(matched["Borehole_ID"].dropna().unique())
            ),
            "Exact_Source_Descriptions": " || ".join(
                sorted(matched["Source_Lithology"].dropna().unique())
            ),
            "Interpretation": (
                "Literal description found"
                if len(matched) else
                "No literal description found"
            )
        })

        # Save the underlying source rows for auditability
        if len(matched):
            matched["NATMO_Comparison_Unit"] = natmo_unit
            matched["NATMO_Search_Term"] = pattern

            matched.to_csv(
                out / (
                    "G21F7_Matches_"
                    + natmo_unit.replace("-", "_").replace(" ", "_")
                    + "_"
                    + pattern.replace(r"\b", "").replace("\\", "")
                    + ".csv"
                ),
                index=False
            )

audit = pd.DataFrame(records)

audit.to_csv(
    out / "G21F7_NATMO_Literal_Lithology_Match_Audit.csv",
    index=False
)

print("NATMO literal lithology audit:")
display(audit)

print("\nSaved to:")
print(out / "G21F7_NATMO_Literal_Lithology_Match_Audit.csv")


**[orig #345]** Evidence-based interpretations → crosswalk v2  
<sub>✅ ran OK</sub>

In [ ]:

import pandas as pd
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference/"
    "G21F7_NATMO_GSI_Borehole_Comparison"
)

crosswalk_path = folder / "G21F7_NATMO_GSI_Borehole_Crosswalk.csv"
audit_path = folder / "G21F7_NATMO_Literal_Lithology_Match_Audit.csv"

crosswalk = pd.read_csv(crosswalk_path)
audit = pd.read_csv(audit_path)

# Evidence-based interpretations.
decisions = {
    "Recent Sand": {
        "status": "GRANULAR_MATERIAL_DOCUMENTED_UNIT_EQUIVALENCE_UNCONFIRMED",
        "confidence": "Moderate for sand occurrence; low for NATMO unit correlation",
        "decision": "Tentative regional association only",
        "comment": (
            "39 intervals in 19 boreholes contain the literal term sand. "
            "This does not establish depositional age or equivalence to "
            "the NATMO Recent Sand map unit."
        )
    },
    "Limestone": {
        "status": "NO_LITERAL_BOREHOLE_SUPPORT",
        "confidence": "Low for correlation",
        "decision": "No borehole correlation approved",
        "comment": (
            "No accepted interval contains the literal term limestone. "
            "This does not establish district-wide absence."
        )
    },
    "Quartzite-Conglomerate": {
        "status": "QUARTZITE_DOCUMENTED_CONGLOMERATE_UNSUPPORTED",
        "confidence": "High for literal quartzite occurrence; low for map correlation",
        "decision": "Partial lithological support; no spatial correlation approved",
        "comment": (
            "13 intervals in 10 boreholes contain quartzite. "
            "No interval contains conglomerate. The combined NATMO map "
            "category cannot be fully validated from these records."
        )
    },
    "Epidotite (provisional transcription)": {
        "status": "NO_LITERAL_BOREHOLE_SUPPORT",
        "confidence": "Low for correlation",
        "decision": "No borehole correlation approved",
        "comment": (
            "No accepted interval contains the literal term epidotite. "
            "The NATMO legend transcription itself remains provisional."
        )
    }
}

# Add evidence fields without changing the original source columns.
for col in [
    "Borehole_Literal_Audit_Status",
    "Borehole_Literal_Interval_Count",
    "Borehole_Literal_Borehole_Count",
    "Borehole_Literal_Search_Terms",
    "Borehole_Evidence_Confidence",
    "Updated_Conceptual_Model_Decision",
    "Updated_Comments"
]:
    crosswalk[col] = ""

for idx, row in crosswalk.iterrows():
    unit = row["NATMO_unit"]
    decision = decisions.get(unit)

    if decision is None:
        continue

    relevant = audit[audit["NATMO_Unit"] == unit]

    crosswalk.loc[idx, "Borehole_Literal_Audit_Status"] = decision["status"]
    crosswalk.loc[idx, "Borehole_Literal_Interval_Count"] = int(
        relevant["Matching_Intervals"].sum()
    )
    crosswalk.loc[idx, "Borehole_Literal_Borehole_Count"] = int(
        relevant["Unique_Boreholes"].max()
    ) if len(relevant) else 0
    crosswalk.loc[idx, "Borehole_Literal_Search_Terms"] = "; ".join(
        relevant["Search_Term"].astype(str)
    )
    crosswalk.loc[idx, "Borehole_Evidence_Confidence"] = decision["confidence"]
    crosswalk.loc[idx, "Updated_Conceptual_Model_Decision"] = decision["decision"]
    crosswalk.loc[idx, "Updated_Comments"] = decision["comment"]

# Preserve GSI and spatial fields as pending.
crosswalk["GSI_corresponding_unit"] = "Not yet verified"
crosswalk["GSI_source_reference"] = "Pending comparison"
crosswalk["Spatial_correlation"] = "Not evaluated"

output = folder / "G21F7_NATMO_GSI_Borehole_Crosswalk_v2.csv"
crosswalk.to_csv(output, index=False)

print("Updated crosswalk saved:")
print(output)

display(crosswalk[[
    "NATMO_unit",
    "Borehole_Literal_Audit_Status",
    "Borehole_Literal_Interval_Count",
    "Borehole_Literal_Borehole_Count",
    "Borehole_Evidence_Confidence",
    "Updated_Conceptual_Model_Decision"
]])


## 4. GSI Plate IV legend, CRS and reference layers

**[orig #346]** Inventory of the GSI source files  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path

root = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps"
)

print("GSI SOURCE FILE INVENTORY")
print("=" * 75)

if not root.exists():
    raise FileNotFoundError(root)

for p in sorted(root.rglob("*")):
    if p.is_file():
        name = p.name.lower()

        if any(term in name for term in [
            "gsi", "plate", "geological", "alwar",
            "legend", "map", "atlas"
        ]):
            print(f"{p.name}")
            print(f"  Path: {p}")
            print(f"  Size: {p.stat().st_size / 1024:.1f} KB")

print("\nInventory completed.")


**[orig #347]** Display GSI Plate IV; save an inspection copy  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
from PIL import Image
from IPython.display import display

root = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps"
)

source = root / "Actual_Geological_Map" / "Alwar_Plate_IV_Geological_Map_HighRes.png"

if not source.exists():
    raise FileNotFoundError(source)

img = Image.open(source).convert("RGB")
W, H = img.size

print("GSI Plate IV dimensions:", img.size)

# Display the complete map first
display(img)

# Output directory
out = root / "G21F7_GSI_Legend_Inspection"
out.mkdir(parents=True, exist_ok=True)

# Save full map copy for inspection
img.save(out / "GSI_Plate_IV_Inspection_Copy.png")

# Display a 2x2 grid of image quadrants for locating the legend.
quadrants = {
    "Top_Left": (0, 0, W//2, H//2),
    "Top_Right": (W//2, 0, W, H//2),
    "Bottom_Left": (0, H//2, W//2, H),
    "Bottom_Right": (W//2, H//2, W, H)
}

for name, bounds in quadrants.items():
    crop = img.crop(bounds)
    path = out / f"GSI_Plate_IV_{name}.png"
    crop.save(path)

    print(f"\n{name}: {crop.size}")
    display(crop)

print("\nInspection images saved in:", out)


**[orig #348]** GSI Plate IV legend register  
<sub>✅ ran OK</sub>

In [ ]:
import os
import pandas as pd

base = (
    "/content/drive/MyDrive/gw_framework/"
    "MODFLOW_Alwar/Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/G21F7_GSI_Legend_Inspection"
)

os.makedirs(base, exist_ok=True)

records = [
    ["Geology", "Alluvium and wind blown sand", "High"],
    ["Geology", "Ajabgarh Group", "High"],
    ["Geology", "Alwar Group", "High"],
    ["Geology", "Raialo Group", "High"],
    ["Geology", "Mangalwar Complex", "High"],
    ["Geology", "Bairath/Dadikar/Harsora/Chapoli/"
     "Ajitgarh/Ajmer Granite etc.", "High"],

    ["Structural Feature", "Fault", "High"],
    ["Structural Feature", "Great Boundary Fault", "High"],
    ["Structural Feature", "Lineament", "High"],

    ["Boundary", "State Boundary", "High"],
    ["Boundary", "District Boundary", "High"],
    ["Boundary", "Block Boundary", "High"],
    ["Boundary", "Basin Boundary", "High"],

    ["Administrative Feature", "District Headquarter", "High"],
    ["Administrative Feature", "Block Headquarter", "High"],
    ["Administrative Feature", "Town", "High"],
]

df = pd.DataFrame(records, columns=[
    "Feature_Category",
    "Literal_Legend_Entry",
    "Transcription_Confidence"
])

df["Source_Map"] = "GSI Plate IV - Alwar Geological Map"
df["Source_Attribution"] = "District Resource Map of Rajasthan - GSI"
df["Interpretation_Status"] = "Literal transcription only"
df["Hydraulic_Property_Assigned"] = "No"
df["MODFLOW_Layer_Assigned"] = "No"
df["Spatial_Correlation_Status"] = "Pending georeferencing"

out = os.path.join(base, "G21F7_GSI_Plate_IV_Legend_Register.csv")
df.to_csv(out, index=False)

print("Saved:", out)
print("Legend entries:", len(df))
display(df)

**[orig #349]** G21F8 — Plate IV CRS and map metadata (text search)  
<sub>✅ ran OK</sub>

In [ ]:
# G21F8 — GSI Plate IV CRS and map metadata investigation

!pip -q install pymupdf

import fitz
import os
import re
import pandas as pd

pdf_path = (
    "/content/drive/MyDrive/gw_framework/"
    "MODFLOW_Alwar/Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/"
    "Alwar_Hydrogeological_Atlas_2013.pdf"
)

out_dir = (
    "/content/drive/MyDrive/gw_framework/"
    "MODFLOW_Alwar/Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/G21F7_GSI_Legend_Inspection"
)

os.makedirs(out_dir, exist_ok=True)

doc = fitz.open(pdf_path)

print("PDF pages:", len(doc))
print("\nPDF METADATA")
for key, value in doc.metadata.items():
    print(f"{key}: {value}")

# Plate IV is PDF page 8 (zero-based index 7)
page = doc[7]
page_text = page.get_text("text")

print("\nPLATE IV EXTRACTED TEXT")
print(page_text[:12000])

# Search for possible CRS / projection / datum references
patterns = [
    r"projection",
    r"projected",
    r"datum",
    r"coordinate system",
    r"polyconic",
    r"UTM",
    r"Everest",
    r"WGS.?84",
    r"modified",
    r"survey of india",
    r"scale",
]

matches = []
for pattern in patterns:
    found = re.findall(
        rf".{{0,100}}{pattern}.{{0,150}}",
        page_text,
        flags=re.IGNORECASE
    )
    for item in found:
        matches.append({
            "Search_Term": pattern,
            "Extracted_Context": item.strip()
        })

match_df = pd.DataFrame(matches)

print("\nPOTENTIAL CRS REFERENCES")
if len(match_df):
    display(match_df)
else:
    print("No CRS-related terms found in extracted Plate IV text.")

# Save the extracted text and audit
text_path = os.path.join(out_dir, "G21F8_Plate_IV_Extracted_Text.txt")
with open(text_path, "w", encoding="utf-8") as f:
    f.write(page_text)

csv_path = os.path.join(out_dir, "G21F8_CRS_Text_Search_Audit.csv")
match_df.to_csv(csv_path, index=False)

print("\nSaved:")
print(text_path)
print(csv_path)

**[orig #350]** G21F9 — search Drive for Rajasthan geological GIS layers  
<sub>✅ ran OK</sub>

In [ ]:
# G21F9 — Search existing Drive for Rajasthan geological GIS reference layers

import os

root = "/content/drive/MyDrive/gw_framework"

keywords = [
    "geology", "geological", "geomorph",
    "alwar", "district", "shapefile",
    "aquifer", "groundwater_atlas"
]

extensions = (
    ".shp", ".gpkg", ".geojson", ".tif",
    ".tiff", ".kml", ".gdb", ".zip"
)

results = []

for current_root, dirs, files in os.walk(root):
    # Avoid searching temporary/cache directories
    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".ipynb_checkpoints"]
    ]

    for filename in files:
        lower = filename.lower()

        if lower.endswith(extensions):
            if any(k in lower for k in keywords):
                results.append(
                    os.path.join(current_root, filename)
                )

print("Potential GIS reference files:", len(results))

for path in results[:200]:
    print(path)

if not results:
    print(
        "\nNo matching GIS reference layers found "
        "inside gw_framework."
    )

**[orig #351]** G21F10 — inspect the existing Alwar spatial reference layers  
<sub>✅ ran OK</sub>

In [ ]:
# G21F10 — Inspect existing Alwar spatial reference layers

!pip -q install rasterio geopandas

import os
import rasterio
import geopandas as gpd

files = {
    "DEM": (
        "/content/drive/MyDrive/gw_framework/"
        "inputs/alwar/dem_alwar_utm43n_30m.tif"
    ),
    "Historical block candidates": (
        "/content/drive/MyDrive/gw_framework/"
        "data/gwl/diagnostics/Alwar_Source_Recovery/"
        "Step_14_Spatial_Validation/"
        "08_historical_alwar_block_candidates.geojson"
    ),
    "Historical block union": (
        "/content/drive/MyDrive/gw_framework/"
        "data/gwl/diagnostics/Alwar_Source_Recovery/"
        "Step_14D_Historical_Boundary_QC/"
        "01_reconstructed_historical_alwar_block_union.geojson"
    )
}

for name, path in files.items():

    print("\n" + "="*70)
    print(name)
    print("="*70)

    if not os.path.exists(path):
        print("FILE NOT FOUND")
        continue

    if path.endswith(".tif"):
        with rasterio.open(path) as src:
            print("CRS:", src.crs)
            print("Bounds:", src.bounds)
            print("Resolution:", src.res)
            print("Dimensions:", src.width, src.height)
            print("NoData:", src.nodata)

    elif path.endswith(".geojson"):
        gdf = gpd.read_file(path)

        print("CRS:", gdf.crs)
        print("Features:", len(gdf))
        print("Geometry types:", gdf.geom_type.value_counts().to_dict())
        print("Bounds:", gdf.total_bounds)
        print("Columns:", list(gdf.columns))

        display(gdf.head(3))

**[orig #352]** G21F11 — DEM and historical boundary spatial validation  
<sub>✅ ran OK</sub>

In [ ]:
# G21F11 — DEM and historical boundary spatial validation

import os
import rasterio
import geopandas as gpd
from shapely.geometry import box

dem_path = (
    "/content/drive/MyDrive/gw_framework/"
    "inputs/alwar/dem_alwar_utm43n_30m.tif"
)

boundary_path = (
    "/content/drive/MyDrive/gw_framework/"
    "data/gwl/diagnostics/Alwar_Source_Recovery/"
    "Step_14D_Historical_Boundary_QC/"
    "01_reconstructed_historical_alwar_block_union.geojson"
)

with rasterio.open(dem_path) as src:
    dem_crs = src.crs
    dem_bounds = src.bounds
    dem_polygon = gpd.GeoDataFrame(
        geometry=[box(*dem_bounds)],
        crs=dem_crs
    )

boundary = gpd.read_file(boundary_path)

# Transform boundary into DEM CRS
boundary_utm = boundary.to_crs(dem_crs)

# Spatial tests
dem_footprint = dem_polygon.geometry.iloc[0]
study_geometry = boundary_utm.geometry.iloc[0]

print("DEM CRS:", dem_crs)
print("Boundary original CRS:", boundary.crs)
print("Boundary transformed CRS:", boundary_utm.crs)

print("\nSPATIAL COVERAGE")
print("Boundary valid:", study_geometry.is_valid)
print("DEM intersects boundary:", dem_footprint.intersects(study_geometry))
print("DEM covers complete boundary:", dem_footprint.covers(study_geometry))

print("\nAREA")
print("Historical boundary area (km²):",
      round(study_geometry.area / 1e6, 3))

print("DEM footprint area (km²):",
      round(dem_footprint.area / 1e6, 3))

print("\nBOUNDARY UTM EXTENT")
print(boundary_utm.total_bounds)

print("\nDEM EXTENT")
print(dem_bounds)

# Save validation report
out_dir = os.path.dirname(boundary_path)

report = {
    "DEM_CRS": str(dem_crs),
    "Boundary_Original_CRS": str(boundary.crs),
    "Boundary_Transformed_CRS": str(boundary_utm.crs),
    "Boundary_Area_km2": study_geometry.area / 1e6,
    "DEM_Area_km2": dem_footprint.area / 1e6,
    "DEM_Intersects_Boundary": dem_footprint.intersects(study_geometry),
    "DEM_Covers_Boundary": dem_footprint.covers(study_geometry)
}

import pandas as pd

report_path = os.path.join(
    out_dir, "G21F11_DEM_Boundary_Spatial_Validation.csv"
)

pd.DataFrame([report]).to_csv(report_path, index=False)

print("\nSaved:", report_path)

## 5. Rajasthan aquifer mapping report maps (in progress)

**[orig #353]** G21F12 — extract the report pages with the two Rajasthan maps (set the PDF path first)  
<sub>✅ ran OK</sub>

In [ ]:
# G21F12 — Extract report pages containing the two Rajasthan maps

!pip -q install pymupdf

import pymupdf
import os
from IPython.display import display, Image

pdf_path = (
    "/content/drive/MyDrive/gw_framework/"
    "MODFLOW_Alwar/Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference/"
    "National Atlas & Thematic Mapping Organisation.pdf"
)

# IMPORTANT:
# Replace this path with the actual path of the
# Rajasthan Ground Water Department final report PDF.
report_path = (
    "/content/drive/MyDrive/gw_framework/"
    "YOUR_REPORT_FOLDER/YOUR_RAJASTHAN_GROUNDWATER_REPORT.pdf"
)

out_dir = (
    "/content/drive/MyDrive/gw_framework/"
    "MODFLOW_Alwar/Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/G21F12_Rajasthan_Report_Maps"
)

os.makedirs(out_dir, exist_ok=True)

if not os.path.exists(report_path):
    print("Report PDF not found. Please update report_path.")
else:
    doc = pymupdf.open(report_path)

    print("PDF pages:", len(doc))

    # Printed page 29 and 38 are likely PDF page indices
    # 28 and 37 if there is no front-matter offset.
    for printed_page, page_index in [(29, 28), (38, 37)]:

        if page_index >= len(doc):
            print("Page index unavailable:", page_index)
            continue

        page = doc[page_index]
        pix = page.get_pixmap(matrix=pymupdf.Matrix(3, 3), alpha=False)

        output = os.path.join(
            out_dir,
            f"G21F12_Figure_Page_{printed_page}.png"
        )

        pix.save(output)

        print("\nPrinted page:", printed_page)
        print("PDF page index:", page_index)
        print("Extracted text:")
        print(page.get_text("text")[:2500])
        print("Saved:", output)

        display(Image(filename=output))

**[orig #354]** Search Drive for the report PDF  
<sub>✅ ran OK</sub>

In [ ]:
!pip -q install pymupdf

import os
import pymupdf
import re

DRIVE_ROOT = "/content/drive/MyDrive"

keywords = [
    "aquifer mapping",
    "village wise groundwater",
    "groundwater potential",
    "ground water potential",
    "strategy for development",
    "rajasthan groundwater",
    "rajasthan ground water"
]

candidates = []

print("Searching Google Drive for likely report PDFs...")

for root, dirs, files in os.walk(DRIVE_ROOT):

    dirs[:] = [
        d for d in dirs
        if d not in [
            ".ipynb_checkpoints",
            "__pycache__",
            ".Trash"
        ]
    ]

    for filename in files:
        if filename.lower().endswith(".pdf"):
            name = filename.lower()

            if any(k in name for k in keywords):
                candidates.append(os.path.join(root, filename))

print("\nFilename matches:", len(candidates))

# Also inspect the text of likely PDFs
for path in candidates:

    print("\n" + "="*80)
    print("FILE:", path)

    try:
        doc = pymupdf.open(path)

        text = ""
        for page in doc[:min(5, len(doc))]:
            text += page.get_text()

        text_lower = text.lower()

        title_match = (
            "aquifer mapping" in text_lower
            and (
                "village wise" in text_lower
                or "groundwater potential" in text_lower
                or "ground water potential" in text_lower
            )
        )

        print("Pages:", len(doc))
        print("Title-like content match:", title_match)

        if title_match:
            print("\n*** POSSIBLE TARGET REPORT FOUND ***")
            print(text[:2500])

    except Exception as e:
        print("Could not inspect:", e)

print("\nSearch complete.")

**[orig #355]** Download the Rajasthan aquifer mapping report and locate the map pages  
<sub>⚠ last run ended with ConnectTimeout</sub>

In [ ]:
!pip -q install pymupdf requests

import os
import requests
import pymupdf
from IPython.display import display, Image

url = (
    "https://phedwater.rajasthan.gov.in/content/dam/doitassets/"
    "water/Ground%20Water/Pdf/Reports/Aquifer-Mapping/"
    "AM%20Final%20Report.pdf"
)

out_dir = (
    "/content/drive/MyDrive/gw_framework/"
    "MODFLOW_Alwar/Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/G21F12_Rajasthan_Report_Maps"
)

os.makedirs(out_dir, exist_ok=True)

pdf_path = os.path.join(
    out_dir, "Rajasthan_Aquifer_Mapping_Final_Report.pdf"
)

# Download only if not already present
if not os.path.exists(pdf_path):
    response = requests.get(url, timeout=120)
    response.raise_for_status()

    with open(pdf_path, "wb") as f:
        f.write(response.content)

    print("Downloaded:", pdf_path)
else:
    print("Using existing PDF:", pdf_path)

doc = pymupdf.open(pdf_path)
print("Total PDF pages:", len(doc))

# Locate pages using their actual extracted text
targets = ["Fig.13", "Fig.19"]

for target in targets:
    print("\nSearching for:", target)

    matches = []

    for i, page in enumerate(doc):
        text = page.get_text("text")

        if target.lower() in text.lower():
            matches.append(i)

    print("Matching PDF page indexes:", matches)

    for page_index in matches:
        page = doc[page_index]

        pix = page.get_pixmap(
            matrix=pymupdf.Matrix(3, 3),
            alpha=False
        )

        image_path = os.path.join(
            out_dir,
            f"{target.replace('.', '')}_PDF_Page_{page_index+1}.png"
        )

        pix.save(image_path)

        print("PDF page:", page_index + 1)
        print("Image saved:", image_path)
        print(page.get_text("text")[:1500])

        display(Image(filename=image_path))

print("\nExtraction complete.")

**[orig #356]** G21F13 — extract the original report maps  
<sub>⚠ last run ended with ValueError</sub>

In [ ]:

# G21F13 — Extract original Rajasthan report maps

from google.colab import drive
drive.mount('/content/drive')

!pip -q install pymupdf

import os
import glob
import pymupdf
import pandas as pd
from IPython.display import display, Image

# --------------------------------------------------
# 1. Folder
# --------------------------------------------------

folder = (
    "/content/drive/MyDrive/gw_framework/"
    "MODFLOW_Alwar/Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/"
    "G21F13_Rajasthan_Geological_Reference"
)

os.makedirs(folder, exist_ok=True)

print("Folder:", folder)

# --------------------------------------------------
# 2. Find PDF
# --------------------------------------------------

pdf_files = glob.glob(os.path.join(folder, "*.pdf"))

print("\nPDF files found:", len(pdf_files))

for f in pdf_files:
    print(os.path.basename(f))

if not pdf_files:
    raise FileNotFoundError(
        "No PDF found. Check that the report was uploaded "
        "to the G21F13_Rajasthan_Geological_Reference folder."
    )

if len(pdf_files) > 1:
    print("\nMultiple PDFs found. Using the first one:")

pdf_path = pdf_files[0]

print("\nSelected PDF:", pdf_path)

# --------------------------------------------------
# 3. Open and identify pages
# --------------------------------------------------

doc = pymupdf.open(pdf_path)

print("Total PDF pages:", len(doc))

targets = {
    "Figure_13_SOI_State_Map": [
        "fig.13",
        "geo-referenced soi state map"
    ],
    "Figure_19_Rajasthan_Geological_Map": [
        "fig.19",
        "geological map of rajasthan"
    ]
}

audit = []
found_pages = {}

for i, page in enumerate(doc):
    text = page.get_text("text").lower()

    for figure_name, phrases in targets.items():

        if all(p in text for p in phrases):
            found_pages[figure_name] = i
            audit.append({
                "Figure": figure_name,
                "PDF_Page_Index": i,
                "PDF_Page_Number": i + 1,
                "Caption_Match": "Confirmed"
            })

print("\nIDENTIFIED FIGURES")

for name, index in found_pages.items():
    print(name, "-> PDF page", index + 1)

# --------------------------------------------------
# 4. Export high-resolution images
# --------------------------------------------------

for figure_name, index in found_pages.items():

    page = doc[index]

    # 400 DPI approximately
    scale = 400 / 72

    pix = page.get_pixmap(
        matrix=pymupdf.Matrix(scale, scale),
        alpha=False
    )

    output = os.path.join(
        folder,
        f"G21F13_{figure_name}_400dpi.png"
    )

    pix.save(output)

    print("\nSaved:", output)
    print("Image dimensions:", pix.width, "x", pix.height)

    # Display resized preview only
    display(Image(filename=output, width=950))

# --------------------------------------------------
# 5. Save page audit
# --------------------------------------------------

audit_path = os.path.join(
    folder,
    "G21F13_Figure_Page_Extraction_Audit.csv"
)

pd.DataFrame(audit).to_csv(audit_path, index=False)

print("\nAudit saved:", audit_path)

if len(found_pages) < 2:
    print(
        "\nWARNING: Both figures were not identified. "
        "We will inspect the report's page text next."
    )

print("\nG21F13 extraction finished.")
